# Code Reviewer Thesis — Experiment Findings

This notebook loads Phase 3 (governance experiment) and Phase 4 (evaluation) results and presents the main findings: primary metrics, McNemar tests, hypotheses, and optional visualizations.

**Data:** DiverseVul C/C++ test set. **Models:** ML (CodeBERT), PaC (Semgrep with p/c + p/cwe-top-25 + custom CWE rules), Hybrid (α·ML + β·PaC with tuned thresholds).

In [ ]:
import json
import os

import pandas as pd

# Paths: run from repo root or from notebooks/
cwd = os.getcwd()
PROJECT_ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == 'notebooks' else cwd
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
PHASE3_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'phase3_results')

report_path = os.path.join(RESULTS_DIR, 'phase4_evaluation_report.json')
if not os.path.isfile(report_path):
    report_path = os.path.join(PHASE3_RESULTS_DIR, 'phase4_evaluation_report.json')
config_path = os.path.join(RESULTS_DIR, 'phase3_hybrid_config.json')
if not os.path.isfile(config_path):
    config_path = os.path.join(PHASE3_RESULTS_DIR, 'phase3_hybrid_config.json')
csv_path = os.path.join(RESULTS_DIR, 'phase3_experiment_results.csv')
if not os.path.isfile(csv_path):
    csv_path = os.path.join(PHASE3_RESULTS_DIR, 'phase3_experiment_results.csv')

with open(report_path) as f:
    report = json.load(f)
with open(config_path) as f:
    hybrid_config = json.load(f)

print('Loaded:', report_path, config_path)

## 1. Hybrid configuration (from Phase 3 tuning)

Validation tuning chose α (ML weight) and β (PaC weight) and thresholds for Block/Review.

In [ ]:
pd.DataFrame([hybrid_config]).T.rename(columns={0: 'value'})

**Interpretation:** If β=0, the hybrid is effectively ML-only (PaC score is ignored in the risk).

## 2. Primary metrics (Block decision)

Precision, Recall, F1 for ML-Only, PaC-Only, and Hybrid.

In [ ]:
pm = report['primary_metrics']
df_metrics = pd.DataFrame(pm).T
df_metrics = df_metrics.rename(columns={'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1'})
df_metrics.round(4)

In [ ]:
%matplotlib inline
ax = df_metrics[['Precision', 'Recall', 'F1']].plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_ylabel('Score')
ax.set_title('Primary metrics by system')
ax.legend(loc='lower right')
ax.set_ylim(0, 1)

## 3. ROC AUC

In [ ]:
auc = report['roc_auc']
pd.DataFrame([auc]).T.rename(columns={0: 'AUC'}).round(4)

## 4. McNemar (Hybrid vs ML, Hybrid vs PaC)

p-value and odds ratio for paired Block decisions.

In [ ]:
mcn = report['mcnemar']
mcn_df = pd.DataFrame({
    'Comparison': list(mcn.keys()),
    'McNemar p': [mcn[k]['mcnemar_p'] for k in mcn],
    'Odds ratio': [mcn[k]['odds_ratio'] for k in mcn],
})
mcn_df.round(4)

## 5. Hypotheses

In [ ]:
hyp = report['hypotheses']
rows = []
for k, v in hyp.items():
    rows.append({'Hypothesis': k, 'Statement': v.get('statement', ''), 'Supported': v.get('supported', None)})
pd.DataFrame(rows)

## 6. Experiment data snapshot (optional)

Label distribution and PaC score distribution from the Phase 3 CSV (sample if large).

In [ ]:
if os.path.isfile(csv_path):
    df = pd.read_csv(csv_path, usecols=['label', 'ml_confidence', 'pac_score', 'decision_ml', 'decision_pac', 'decision_hybrid'], nrows=100_000)
    print('Sample size:', len(df))
    print('Label distribution:')
    print(df['label'].value_counts())
    print('\nPaC score > 0:', (df['pac_score'] > 0).sum(), f'({100 * (df["pac_score"] > 0).mean():.1f}%)')
    print('\nDecision counts (Block=2):')
    for col in ['decision_ml', 'decision_pac', 'decision_hybrid']:
        if col in df.columns:
            print(f'  {col}:', df[col].value_counts().to_dict())
else:
    print('Phase 3 CSV not found at', csv_path)

## 7. Summary / findings

- **PaC** now has non-zero F1 (custom CWE rules + p/c + p/cwe-top-25); precision is higher than ML but recall is low.
- **Hybrid** in this run equals ML because tuning set β=0 (validation F1 was best with ML-only).
- **H2** (Hybrid recall > PaC recall) is supported; **H1** and **H3** are not in this configuration.
- For the thesis: report these metrics and discuss that PaC adds signal but the current tuning did not improve over ML-only; possible future work: different combination strategies or thresholds.